# Validación Entorno Inicial

Ejecuta la siguiente celda antes del preprocesamiento para confirmar qué intérprete y carpeta está usando el notebook, y comprobar que las librerías necesarias estén disponibles.

In [ ]:
import sys
from pathlib import Path

# El intérprete que ejecuta ESTE notebook. Si la ruta no contiene .venv,
# el kernel es el equivocado: Kernel -> Change Kernel.
print("Intérprete:", sys.executable)

# La carpeta desde la que se ejecuta. Las rutas relativas del notebook
# se resuelven desde aquí, no desde donde está el archivo .ipynb.
print("Carpeta de trabajo:", Path.cwd())

# Las librerías del proyecto, con su versión
for lib in ["numpy", "pandas", "matplotlib", "sklearn"]:
    try:
        print(f" {lib:12}", __import__(lib).__version__)
    except ImportError:
        print(f" {lib:12} NO DISPONIBLE")

# Fase 2: exploración y preprocesamiento

Este notebook carga el archivo original, revisa su calidad y prepara una copia de trabajo con fechas y montos convertidos. El archivo CSV original se conserva sin modificaciones.

**Recorrido:** `CSV original` → `revisión de estructura` → `inspección de valores` → `conversiones controladas` → `resúmenes para validar`

En cada bloque, las tablas y conteos permiten comprobar qué se leyó o transformó antes de continuar.

## 1. Cargar el archivo y comprobar la lectura

`CSV` → `DataFrame df` → `dimensiones y muestra`

Se importa `pandas`, se define la ruta relativa al archivo y se lee el CSV detectando su separador y usando la codificación indicada. La ejecución muestra cuántas filas y columnas se cargaron y las primeras cinco filas para revisar encabezados y caracteres.

In [ ]:
# PASO 1: Importar las herramientas necesarias.
# Path permite trabajar con rutas de carpetas y archivos.
from pathlib import Path

# pandas permite cargar y analizar datos en tablas.
import pandas as pd


# PASO 2: Indicar dónde está la base de datos.
# Desde F2/notebooks subimos a la raíz del proyecto.
# Luego entramos a F1/data/raw, donde está el archivo original.
ruta = Path("../../F1/data/raw/04LicitacionesEne.csv")


# PASO 3: Cargar el CSV en una tabla llamada df.
# sep=None intenta detectar el separador del archivo.
# engine="python" permite esa detección automática.
# encoding="latin-1" es la codificación con la que lograste abrirlo.
# Este comando lee el archivo; no modifica el CSV original.
df = pd.read_csv(
    ruta,
    sep=None,
    engine="python",
    encoding="latin-1"
)


# PASO 4: Mostrar la cantidad de filas y columnas.
# shape[0] corresponde a las filas.
# shape[1] corresponde a las columnas.
print("Cantidad de filas:", df.shape[0])
print("Cantidad de columnas:", df.shape[1])


# PASO 5: Mostrar las primeras cinco filas.
# Revisaremos que las columnas y las tildes se lean correctamente.
display(df.head())

## 2. Perfilar columnas y detectar duplicados

`df` → `tipos + nulos + valores distintos + duplicados`

La tabla resume cada columna con su tipo, cantidad y porcentaje de nulos y número de valores únicos. Al final se cuenta cuántas filas son idénticas; solo se reportan, no se eliminan.

In [ ]:
# PASO 6: Crear un resumen de las columnas de la base.
resumen = pd.DataFrame({
    
    # Mostrar el tipo que pandas reconoció en cada columna.
    "Tipo de dato": df.dtypes.astype(str),
    
    # Contar los valores faltantes de cada columna.
    "Valores nulos": df.isna().sum(),
    
    # Calcular el porcentaje de valores faltantes.
    "Porcentaje nulos": (df.isna().mean() * 100).round(2),
    
    # Contar cuántos valores diferentes tiene cada columna.
    "Valores distintos": df.nunique()
})

# PASO 7: Mostrar todas las filas del resumen.
# Este ajuste solo afecta la visualización de esta tabla.
with pd.option_context("display.max_rows", None):
    display(resumen)


# PASO 8: Contar las filas que son completamente iguales.
# Todavía no las eliminamos: primero debemos revisar su significado.
duplicados = df.duplicated().sum()

print("Filas completamente duplicadas:", duplicados)

## 3. Inspeccionar fechas, montos y filas de una licitación

`columnas originales` → `muestras concretas` → `contexto por licitación`

Se muestran ejemplos de las tres fechas y de moneda y montos tal como vienen en el CSV. Luego se toma la primera licitación encontrada y se presentan hasta 15 filas con producto, proveedor y resultado para entender qué representa cada registro.

In [ ]:
# PASO 9: Ver ejemplos de las fechas.
# Primero observamos su formato para convertirlas correctamente después.
columnas_fecha = [
    "FechaPublicacion",
    "FechaCierre",
    "FechaAdjudicacion"
]

display(df[columnas_fecha].head(10))


# PASO 10: Revisar cómo están escritos los montos y sus monedas.
# Todavía no reemplazamos puntos ni comas.
columnas_monto = [
    "MonedaOferta",
    "MontoNetoOferta",
    "MontoTotalOferta"
]

display(df[columnas_monto].head(10))


# PASO 11: Examinar varias filas de una misma licitación.
# Esto ayuda a entender qué representa cada fila de la base.
licitacion_ejemplo = df["NroLicitacion"].iloc[0]

columnas_detalle = [
    "NroLicitacion",
    "CodigoProductoONU",
    "NombreItem",
    "ProveedorRUT",
    "EstadoOferta",
    "ResultadoOferta"
]

display(
    df.loc[
        df["NroLicitacion"] == licitacion_ejemplo,
        columnas_detalle
    ].head(15)
)

## 4. Convertir fechas en una copia de trabajo

`df original` → `df_trabajo` → `fechas datetime + control de fallos`

Se conserva `df` y se crea `df_trabajo` para transformar las fechas sin modificar los datos cargados. La tabla de control separa los nulos que ya existían de los valores presentes que no coinciden con el formato esperado; esos fallos quedan como `NaT`.

In [ ]:
# PASO 12: Crear una copia para trabajar.
# df conserva los datos tal como se cargaron.
df_trabajo = df.copy()


# PASO 13: Indicar las columnas de fecha que revisamos.
columnas_fecha = [
    "FechaPublicacion",
    "FechaCierre",
    "FechaAdjudicacion"
]


# PASO 14: Convertir el texto a fechas.
# %d: día; %m: mes; %Y: año; %H: hora; %M: minutos.
# errors="coerce" convierte los valores no interpretables en NaT.
# Después contamos esos casos para no perderlos de vista.
revision_fechas = []

for columna in columnas_fecha:
    original = df[columna]

    df_trabajo[columna] = pd.to_datetime(
        original,
        format="%d-%m-%Y %H:%M",
        errors="coerce"
    )

    # Detectar valores que existían, pero no pudieron convertirse.
    fallos = original.notna() & df_trabajo[columna].isna()

    revision_fechas.append({
        "Columna": columna,
        "Nulos originales": original.isna().sum(),
        "Fallos de conversión": fallos.sum()
    })


# PASO 15: Mostrar el control de la conversión.
display(pd.DataFrame(revision_fechas))

# Mostrar ejemplos de las fechas convertidas.
display(df_trabajo[columnas_fecha].head())

## 5. Revisar monedas y formatos de montos

`MonedaOferta` → `frecuencias`  |  `montos` → `conteo y ejemplos especiales`

Primero se cuentan las monedas, incluidos los valores nulos. Después se inspeccionan los montos como texto y se listan ejemplos que contienen algo distinto de dígitos, como separadores o espacios. Esta revisión es diagnóstica: todavía no cambia los datos.

In [ ]:
# PASO 16: Revisar las monedas presentes.
# Los montos de monedas diferentes deben analizarse por separado.
display(df_trabajo["MonedaOferta"].value_counts(dropna=False))


# PASO 17: Buscar montos con puntos, comas u otros caracteres.
# Esta revisión no modifica los valores.
for columna in ["MontoNetoOferta", "MontoTotalOferta"]:

    # Convertir temporalmente a texto y quitar espacios exteriores.
    texto = df_trabajo[columna].astype("string").str.strip()

    # Identificar valores que no son únicamente dígitos.
    tiene_formato_especial = ~texto.str.fullmatch(
        r"\d+", na=False
    ) & texto.notna()

    print("\nColumna:", columna)
    print("Valores con formato especial:", tiene_formato_especial.sum())

    # Mostrar hasta 20 ejemplos distintos para revisar el formato.
    display(
        texto[tiene_formato_especial]
        .drop_duplicates()
        .head(20)
        .to_frame(name="Ejemplos")
    )

## 6. Convertir montos a valores numéricos

`texto con coma decimal` → `número` → `reporte de fallos y tipos`

En `df_trabajo`, la coma se reemplaza por punto y se intenta convertir cada monto a número. Los valores que no se puedan interpretar se convierten en nulos y se reportan, distinguiéndolos de los nulos originales. Los puntos existentes se conservan; por eso conviene revisar los formatos especiales identificados en el bloque anterior.

In [ ]:
# PASO 18: Convertir los montos de texto a número.
# Trabajamos en df_trabajo; df conserva los valores originales.
revision_montos = []

for columna in ["MontoNetoOferta", "MontoTotalOferta"]:

    # Conservar el texto para controlar la conversión.
    original = df_trabajo[columna].astype("string").str.strip()

    # Cambiar la coma decimal por un punto.
    # No eliminamos puntos ni redondeamos los montos.
    texto_convertido = original.str.replace(",", ".", regex=False)

    # Convertir a número.
    # Los valores no interpretables quedan como nulos y se reportan.
    numeros = pd.to_numeric(texto_convertido, errors="coerce")

    # Detectar valores presentes que no pudieron convertirse.
    fallos = original.notna() & numeros.isna()

    revision_montos.append({
        "Columna": columna,
        "Nulos originales": original.isna().sum(),
        "Fallos de conversión": fallos.sum()
    })

    # Mostrar ejemplos si se detectan problemas.
    if fallos.any():
        print("Revisar estos valores de:", columna)
        display(original[fallos].drop_duplicates().head(20))

    # Guardar el resultado numérico en la copia de trabajo.
    df_trabajo[columna] = numeros


# PASO 19: Mostrar el control de la conversión.
display(pd.DataFrame(revision_montos))

# Comprobar los tipos de datos resultantes.
display(
    df_trabajo[["MontoNetoOferta", "MontoTotalOferta"]].dtypes
)

## 7. Validar montos y revisar ofertas

`monto total` → `estadísticos por moneda` → `ceros/negativos` → `mayores ofertas CLP`

Se resumen cantidad, mínimo, mediana, promedio y máximo por moneda, sin mezclar unidades. Después se cuentan montos cero y negativos y se muestran las cinco ofertas más altas en CLP con sus datos de contexto. Son señales para revisar, no reglas automáticas para borrar registros.

In [ ]:
# PASO 20: Resumir los montos por moneda.
# No mezclamos CLP, CLF, USD y UTM.
resumen_montos = (
    df_trabajo.groupby("MonedaOferta")["MontoTotalOferta"]
    .agg(
        Cantidad="count",
        Minimo="min",
        Mediana="median",
        Promedio="mean",
        Maximo="max"
    )
)

display(resumen_montos)


# PASO 21: Contar montos iguales a cero y negativos.
# Solo los identificamos; todavía no eliminamos ninguna fila.
for columna in ["MontoNetoOferta", "MontoTotalOferta"]:
    print("\nColumna:", columna)
    print("Montos iguales a cero:", (df_trabajo[columna] == 0).sum())
    print("Montos negativos:", (df_trabajo[columna] < 0).sum())


# PASO 22: Revisar las cinco ofertas de mayor monto en CLP.
# Un monto alto no es necesariamente un error.
ofertas_clp = df_trabajo[
    df_trabajo["MonedaOferta"] == "CLP"
]

display(
    ofertas_clp.nlargest(5, "MontoTotalOferta")[
        [
            "NroLicitacion",
            "NombreItem",
            "CantidadOferta",
            "MontoNetoOferta",
            "MontoTotalOferta",
            "ResultadoOferta"
        ]
    ]
)